# Base cohort selection: AoU premade ancestry labels

Replaces the entire 1000G-reference-projection cascade this pipeline used to run
(`submit_pca_r1.ipynb` -> `round1_filter.ipynb` -> `submit_pca_r2.ipynb`/
`round2_filter.ipynb` -> `reverse_pca_aou.ipynb`) for base-cohort ancestry selection.
That whole cascade existed to classify AoU samples against 1000G population labels;
this notebook skips classification entirely and uses AoU's own genomic ancestry
classifier output instead -- a continental-level auxiliary flat file (not a BigQuery
table), one row per participant, `ancestry_pred` column with 6 continental groups
(`afr`, `amr`, `eas`, `eur`, `mid`, `sas`). Confirmed via a real `ls` of the v9 CDR
mount: `wgs/short_read/snpindel/aux/ancestry/ancestry_preds.tsv`.

Writes one keep-list per `BASE_GROUP` (`eur`, `afr`) -- every downstream sample set
that shares a `BASE_GROUP` (`eur`/`eur_stringent`/`eur_loose`/`eur_premade_label` all
share `eur`; `afr` is its own base group) reads the SAME merged/QC'd/LD-pruned ACAF
panel (`03_genome_wide_qc_thinning_merge.ipynb`) and the SAME PCA fit
(`05_final_pca.ipynb`) -- only `05_final_pca.ipynb`'s own self-referential Mahalanobis
threshold differs between them. Building one shared panel/PCA per base group instead
of five separate ones (the old per-`SAMPLE_SET` convention) is both simpler and
avoids redundant QC/pruning/PCA work across sample sets that ultimately draw from
the identical starting population.

v9's path is confirmed directly. v8's is **not** -- this file didn't exist when this
repo last ran v8, so its path below is inferred by analogy to `submit_pca_r1.ipynb`'s
`CDR_ACAF_PGEN_DIR` v8 branch, not independently confirmed.

## Inputs

In [ ]:
import os
import pandas as pd

WORKSPACE_BUCKET = os.path.expanduser(
    "~/workspace/Data from All of Us Controlled Tier /shared-env-pilot"
)
AOU_ROOT = os.path.dirname(WORKSPACE_BUCKET)   # only relevant for v8, see markdown above

CDR_VERSION = "v9"

# v9 confirmed via a real `ls` of the mounted CDR; v8 inferred by analogy to
# submit_pca_r1.ipynb's CDR_ACAF_PGEN_DIR v8 branch -- NOT independently confirmed.
CDR_ANCESTRY_PREDS_PATH = {
    "v8": f"{AOU_ROOT}/vwb-aou-datasets-controlled/v8/wgs/short_read/snpindel/aux/ancestry/ancestry_preds.tsv",  # UNCONFIRMED
    "v9": os.path.expanduser(
        "~/workspace/cdrv9/vwb-aou-datasets-controlled-v9/v9/wgs/short_read/snpindel/aux/ancestry/ancestry_preds.tsv"
    ),
}
ancestry_preds_path = CDR_ANCESTRY_PREDS_PATH[CDR_VERSION]
assert os.path.isfile(ancestry_preds_path), f"ancestry_preds.tsv not found: {ancestry_preds_path!r}"

BUCKET_DIR = f"{WORKSPACE_BUCKET}/{CDR_VERSION}/01_ancestry_filtering"
OUT_DIR = f"{BUCKET_DIR}/premade_label_{CDR_VERSION}"
os.makedirs(OUT_DIR, exist_ok=True)

# the two continental groups every downstream SAMPLE_SET maps onto -- see
# 05_final_pca.ipynb's SAMPLE_SETS for the sample_set -> base_group mapping
BASE_GROUPS = ["eur", "afr"]

print(ancestry_preds_path)
print(OUT_DIR)

## Write one keep-list per base group

`.str.lower()`: defensive against `'eur'` vs `'EUR'` casing, not yet confirmed either
way. AoU's own docs use `"research_id"` for this file's person-identifier column
(their term for `person_id` elsewhere in the CDR) -- handled defensively too.

In [ ]:
ancestry_preds = pd.read_csv(ancestry_preds_path, sep="\t")

id_col = "research_id" if "research_id" in ancestry_preds.columns else "person_id"
assert id_col in ancestry_preds.columns and "ancestry_pred" in ancestry_preds.columns, \
    f"unexpected columns in ancestry_preds.tsv: {list(ancestry_preds.columns)}"

print("ancestry_pred value counts:")
print(ancestry_preds["ancestry_pred"].value_counts())

keep_paths = {}
for base_group in BASE_GROUPS:
    ids = ancestry_preds.loc[ancestry_preds["ancestry_pred"].str.lower() == base_group, id_col]
    keep_path = os.path.join(OUT_DIR, f"premade_keep_ids_{base_group}.txt")
    ids.to_csv(keep_path, index=False, header=False)
    keep_paths[base_group] = keep_path
    print(f"[{base_group}] Wrote {len(ids)} IDs to {keep_path}")

## Next steps

`03_genome_wide_qc_thinning_merge.ipynb` (`BASE_GROUP` variable, same values as
`BASE_GROUPS` above) reads `premade_keep_ids_{base_group}.txt` next, to build the
QC'd/LD-pruned ACAF panel each base group's downstream sample sets share.